In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
from pathlib import Path

import zuko
import torch

from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# Dataset preparation

## Natural disaster datasets

### Load and clean the datasets

In [71]:
# Define paths to the storm events CSV files
data_dir = Path('datasets')
files = [
    data_dir / 'StormEvents_details-ftp_v1.0_d2016_c20250818.csv',
    data_dir / 'StormEvents_details-ftp_v1.0_d2017_c20250520.csv',
    data_dir / 'StormEvents_details-ftp_v1.0_d2018_c20250520.csv',
    data_dir / 'StormEvents_details-ftp_v1.0_d2019_c20250520.csv',
    data_dir / 'StormEvents_details-ftp_v1.0_d2020_c20251118.csv',
    data_dir / 'StormEvents_details-ftp_v1.0_d2021_c20250520.csv',
    data_dir / 'StormEvents_details-ftp_v1.0_d2022_c20250721.csv',
]

# Load datasets
print("Loading datasets...")
df_list = []
for file in files:
    # print(f"  Loading {file.name}...")
    df = pd.read_csv(file)
    # print(f"    Shape: {df.shape}")
    df_list.append(df)

# Concatenate the datasets
df = pd.concat(df_list, ignore_index=True)
print(f"\nCombined dataset shape: {df.shape}")

Loading datasets...

Combined dataset shape: (436146, 51)


In [72]:
useless_cols = [
    'BEGIN_YEARMONTH', 'BEGIN_DAY', 'BEGIN_TIME', 
    'END_YEARMONTH', 'END_DAY', 'END_TIME',
    'EPISODE_ID', 'EVENT_ID',
    'SOURCE',
    'YEAR',
    'MONTH_NAME',
    'WFO',
    'CATEGORY',
    'TOR_OTHER_WFO', 'TOR_OTHER_CZ_STATE', 'TOR_OTHER_CZ_FIPS', 'TOR_OTHER_CZ_NAME', 
    'CZ_TYPE', 'CZ_NAME', 'CZ_TIMEZONE',
    'EPISODE_NARRATIVE', 'EVENT_NARRATIVE', 'DATA_SOURCE',
    
    'FLOOD_CAUSE',
    'TOR_LENGTH',
    'TOR_WIDTH',
    'BEGIN_RANGE',
    'BEGIN_AZIMUTH',
    'BEGIN_LOCATION',
    'END_RANGE',
    'END_AZIMUTH',
    'END_LOCATION',
    'END_LAT',
    'END_LON',
    'MAGNITUDE_TYPE',
    'TOR_F_SCALE',
]

df1 = df.drop(columns=useless_cols, inplace=False)

In [73]:
df1.columns

Index(['STATE', 'STATE_FIPS', 'EVENT_TYPE', 'CZ_FIPS', 'BEGIN_DATE_TIME',
       'END_DATE_TIME', 'INJURIES_DIRECT', 'INJURIES_INDIRECT',
       'DEATHS_DIRECT', 'DEATHS_INDIRECT', 'DAMAGE_PROPERTY', 'DAMAGE_CROPS',
       'MAGNITUDE', 'BEGIN_LAT', 'BEGIN_LON'],
      dtype='object')

### Aggregate features

In [74]:
df1['EVENT_TYPE'] = df1['EVENT_TYPE'].replace({
    'Thunderstorm Wind': 'Wind',
    'Hail': 'Hail',
    'Marine Hail': 'Hail',
    'Flood': 'Flood',
    'Flash Flood': 'Flood',
    'Coastal Flood': 'Flood',
    'Lakeshore Flood': 'Flood',
    'Winter Weather': 'Freeze',
    'Extreme Cold/Wind Chill': 'Freeze',
    'Frost/Freeze': 'Freeze',
    'Cold/Wind Chill': 'Freeze',
    'Freezing Fog': 'Freeze',
    'Heat': 'Heat',
    'Excessive Heat': 'Heat',
    'Heavy Snow': 'Snow',
    'Blizzard': 'Snow',
    'Lake-Effect Snow': 'Snow',
    'Sleet': 'Snow',
    'Winter Storm': 'Storm',
    'Marine Thunderstorm Wind': 'Storm',
    'Tornado': 'Storm',
    'Tropical Storm': 'Storm',
    'Dust Storm': 'Storm',
    'Funnel Cloud': 'Storm',
    'Ice Storm': 'Storm',
    'Waterspout': 'Storm',
    'Marine Tropical Storm': 'Storm',
    'Hurricane (Typhoon)': 'Storm',
    'Marine Hurricane/Typhoon': 'Storm',
    'High Wind':  'Wind',
    'Strong Wind':  'Wind',
    'Marine High Wind':  'Wind',
    'Marine Strong Wind':  'Wind',
    'Dense Fog':  'Fog',
    'Marine Dense Fog':  'Fog'
})

In [75]:
def damage_to_numeric(damage_str):
    if pd.isna(damage_str):
        return 0
    multipliers = {'K': 1_000, 'M': 1_000_000, 'B': 1_000_000_000}
    if damage_str[-1] in multipliers:
        try:
            value = float(damage_str[:-1])
            return value * multipliers[damage_str[-1]]
        except ValueError:
            return 0
    try:
        return float(damage_str)
    except ValueError:
        return 0   

df1['DAMAGES'] = df1['DAMAGE_PROPERTY'].apply(damage_to_numeric) + df1['DAMAGE_CROPS'].apply(damage_to_numeric)
df2 = df1.drop(columns=['DAMAGE_PROPERTY', 'DAMAGE_CROPS'], inplace=False)

In [85]:
df2['CASUALTIES'] = df2['INJURIES_DIRECT'].fillna(0) + df2['INJURIES_INDIRECT'].fillna(0) + df2['DEATHS_DIRECT'].fillna(0) + df2['DEATHS_INDIRECT'].fillna(0)
df3 = df2.drop(columns=['INJURIES_DIRECT', 'INJURIES_INDIRECT', 'DEATHS_DIRECT', 'DEATHS_INDIRECT'], inplace=False)

Actually keep only few events type for first experiments

In [86]:
df2['EVENT_TYPE'].unique()

array(['Heavy Rain', 'Wind', 'Storm', 'Heat', 'Flood', 'Drought', 'Hail',
       'Freeze', 'Lightning', 'Wildfire', 'Snow', 'Fog', 'Avalanche',
       'High Surf', 'Debris Flow', 'Astronomical Low Tide', 'Rip Current',
       'Dust Devil', 'Storm Surge/Tide', 'Sneakerwave',
       'Marine Tropical Depression', 'Seiche', 'Tropical Depression',
       'Dense Smoke', 'Volcanic Ashfall', 'Tsunami'], dtype=object)

In [87]:
df3 = df3[df3['EVENT_TYPE'].isin(['Flood', 'Tornado', 'Hail', 'Storm', 'Tsunami', 'Freeze', 'Snow', 'Heat', 'Heavy Rain', 'Wildfire'])]

In [ ]:
df3['EVENT_TYPE'].nunique()

9

### Create unique FIPS

Unique FIPS is created starting from FIPS state (2 numbers) + FIPS county (3 numbers) (fill with zero the entry with less than this numbers)

In [89]:
df3['FIPS'] = df3['STATE_FIPS'].astype(str).str.zfill(2) + df3['CZ_FIPS'].astype(str).str.zfill(3)
print(df3[['FIPS', 'STATE_FIPS', 'CZ_FIPS']].head(10))

     FIPS  STATE_FIPS  CZ_FIPS
0   45091          45       91
6   56001          56        1
7   56012          56       12
8   56013          56       13
9   92247          92      247
10  92263          92      263
14  09001           9        1
15  09013           9       13
19  36093          36       93
23  09013           9       13


In [ ]:
df3.drop(columns=['STATE_FIPS', 'CZ_FIPS'], inplace=True)

### Encode features

In [91]:
# categorial -> numeric
df3['EVENT_TYPE'] = df3['EVENT_TYPE'].astype('category').cat.codes

In [108]:
# EVENT_TYPE one-hot encoding
enc = OneHotEncoder(sparse_output=False)
event_type_encoded = enc.fit_transform(df3[['EVENT_TYPE']])
event_type_df = pd.DataFrame(event_type_encoded, columns=[f"EVENT_TYPE_{cat}" for cat in enc.categories_[0]], index=df3.index)
                             
df4 = pd.concat([df3.drop(columns=['EVENT_TYPE'], inplace=False), event_type_df], axis=1)

In [109]:
df4.head()

,STATE,BEGIN_DATE_TIME,END_DATE_TIME,MAGNITUDE,BEGIN_LAT,BEGIN_LON,DAMAGES,CASUALTIES,FIPS,EVENT_TYPE_0,EVENT_TYPE_1,EVENT_TYPE_2,EVENT_TYPE_3,EVENT_TYPE_4,EVENT_TYPE_5,EVENT_TYPE_6,EVENT_TYPE_7,EVENT_TYPE_8
0,SOUTH CAROLINA,15-JUL-16 17:15:00,15-JUL-16 17:15:00,NaN,34.940,-81.030,2000.0,0,45091,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
6,WYOMING,04-DEC-16 03:00:00,05-DEC-16 06:00:00,NaN,NaN,NaN,0.0,0,56001,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
7,WYOMING,04-DEC-16 04:00:00,05-DEC-16 05:00:00,NaN,NaN,NaN,0.0,0,56012,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
8,WYOMING,04-DEC-16 04:00:00,05-DEC-16 05:00:00,NaN,NaN,NaN,0.0,0,56013,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
9,LAKE SUPERIOR,12-JUL-16 01:30:00,12-JUL-16 01:35:00,41.0,46.969,-88.398,0.0,0,92247,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [110]:
# fill BEGIN_LAT and BEGIN_LON NaN values with the mean of their respective columns
df4['BEGIN_LAT'] = df4['BEGIN_LAT'].fillna(df4['BEGIN_LAT'].mean())
df4['BEGIN_LON'] = df4['BEGIN_LON'].fillna(df4['BEGIN_LON'].mean())

In [111]:
# encode time cols BEGIN_DATE_TIME and END_DATE_TIME
fmt = "%d-%b-%y %H:%M:%S"

df4['BEGIN_DATE_TIME'] = pd.to_datetime(df4['BEGIN_DATE_TIME'], format=fmt, errors='coerce')
df4['END_DATE_TIME'] = pd.to_datetime(df4['END_DATE_TIME'], format=fmt, errors='coerce')

 # Extract hour and day of year
for prefix in ['BEGIN', 'END']:
    dt_col = f'{prefix}_DATE_TIME'
    # Hour of day cyclical encoding
    df4[f'{prefix}_HOUR_sin'] = np.sin(2 * np.pi * df4[dt_col].dt.hour / 24)
    df4[f'{prefix}_HOUR_cos'] = np.cos(2 * np.pi * df4[dt_col].dt.hour / 24)
    # Day of year cyclical encoding
    df4[f'{prefix}_DOY_sin'] = np.sin(2 * np.pi * df4[dt_col].dt.dayofyear / 366)
    df4[f'{prefix}_DOY_cos'] = np.cos(2 * np.pi * df4[dt_col].dt.dayofyear / 366)

# Optionally drop original datetime columns
# df.drop(columns=['BEGIN_DATE_TIME', 'END_DATE_TIME'], inplace=True)

In [112]:
df4.columns

Index(['STATE', 'BEGIN_DATE_TIME', 'END_DATE_TIME', 'MAGNITUDE', 'BEGIN_LAT',
       'BEGIN_LON', 'DAMAGES', 'CASUALTIES', 'FIPS', 'EVENT_TYPE_0',
       'EVENT_TYPE_1', 'EVENT_TYPE_2', 'EVENT_TYPE_3', 'EVENT_TYPE_4',
       'EVENT_TYPE_5', 'EVENT_TYPE_6', 'EVENT_TYPE_7', 'EVENT_TYPE_8',
       'BEGIN_HOUR_sin', 'BEGIN_HOUR_cos', 'BEGIN_DOY_sin', 'BEGIN_DOY_cos',
       'END_HOUR_sin', 'END_HOUR_cos', 'END_DOY_sin', 'END_DOY_cos'],
      dtype='object')

In [113]:
df4.drop(columns=['MAGNITUDE', 'BEGIN_LAT', 'BEGIN_LON'], inplace=True)

In [116]:
nd_df = df4.copy()
nd_df.head()

,STATE,BEGIN_DATE_TIME,END_DATE_TIME,DAMAGES,CASUALTIES,FIPS,EVENT_TYPE_0,EVENT_TYPE_1,EVENT_TYPE_2,EVENT_TYPE_3,...,EVENT_TYPE_7,EVENT_TYPE_8,BEGIN_HOUR_sin,BEGIN_HOUR_cos,BEGIN_DOY_sin,BEGIN_DOY_cos,END_HOUR_sin,END_HOUR_cos,END_DOY_sin,END_DOY_cos
0,SOUTH CAROLINA,2016-07-15 17:15:00,2016-07-15 17:15:00,2000.0,0,45091,0.0,0.0,0.0,0.0,...,0.0,0.0,-0.965926,-0.258819,-0.238033,-0.971257,-0.965926,-2.588190e-01,-0.238033,-0.971257
6,WYOMING,2016-12-04 03:00:00,2016-12-05 06:00:00,0.0,0,56001,0.0,0.0,0.0,0.0,...,0.0,0.0,0.707107,0.707107,-0.447094,0.894487,1.000000,6.123234e-17,-0.431673,0.902030
7,WYOMING,2016-12-04 04:00:00,2016-12-05 05:00:00,0.0,0,56012,0.0,0.0,0.0,0.0,...,0.0,0.0,0.866025,0.500000,-0.447094,0.894487,0.965926,2.588190e-01,-0.431673,0.902030
8,WYOMING,2016-12-04 04:00:00,2016-12-05 05:00:00,0.0,0,56013,0.0,0.0,0.0,0.0,...,0.0,0.0,0.866025,0.500000,-0.447094,0.894487,0.965926,2.588190e-01,-0.431673,0.902030
9,LAKE SUPERIOR,2016-07-12 01:30:00,2016-07-12 01:35:00,0.0,0,92247,0.0,0.0,0.0,0.0,...,0.0,0.0,0.258819,0.965926,-0.187719,-0.982223,0.258819,9.659258e-01,-0.187719,-0.982223


### Normalize features

In [ ]:
scaler = StandardScaler()

## County population datasets

In [125]:
url_county = data_dir / 'cc-est2024-alldata.csv'
cs_df1 = pd.read_csv(url_county, encoding='latin1')

In [126]:
cs_df1.head()

,SUMLEV,STATE,COUNTY,STNAME,CTYNAME,YEAR,AGEGRP,TOT_POP,TOT_MALE,TOT_FEMALE,...,HWAC_MALE,HWAC_FEMALE,HBAC_MALE,HBAC_FEMALE,HIAC_MALE,HIAC_FEMALE,HAAC_MALE,HAAC_FEMALE,HNAC_MALE,HNAC_FEMALE
0,50,1,1,Alabama,Autauga County,1,0,58800,28693,30107,...,963,844,132,116,42,33,22,25,19,10
1,50,1,1,Alabama,Autauga County,1,1,3491,1818,1673,...,93,60,17,11,3,0,11,1,3,0
2,50,1,1,Alabama,Autauga County,1,2,3663,1875,1788,...,92,77,6,9,9,4,0,1,1,2
3,50,1,1,Alabama,Autauga County,1,3,4189,2152,2037,...,94,93,12,13,1,3,2,2,3,1
4,50,1,1,Alabama,Autauga County,1,4,3876,1960,1916,...,79,79,9,11,5,3,2,4,3,2


In [121]:
url_landmass = data_dir / 'county_landmass.csv'
lm_df1 = pd.read_csv(url_landmass)

In [122]:
lm_df1.head()

,FIPS,FIPS_state,FIPS_county,state_abbrev,state,county,sq_mi,land_sq_mi,water_sq_mi
0,1001,1,1,AL,Alabama,Autauga,604.45,595.97,8.48
1,1003,1,3,AL,Alabama,Baldwin,2026.93,1596.35,430.58
2,1005,1,5,AL,Alabama,Barbour,904.52,884.90,19.61
3,1007,1,7,AL,Alabama,Bibb,626.16,623.03,3.14
4,1009,1,9,AL,Alabama,Blount,650.60,645.59,5.02


### Create unique FIPS

In [127]:
cs_df1['FIPS'] = cs_df1['STATE'].astype(str).str.zfill(2) + cs_df1['COUNTY'].astype(str).str.zfill(3)
print(cs_df1[['FIPS', 'STATE', 'COUNTY']].head(10))

    FIPS  STATE  COUNTY
0  01001      1       1
1  01001      1       1
2  01001      1       1
3  01001      1       1
4  01001      1       1
5  01001      1       1
6  01001      1       1
7  01001      1       1
8  01001      1       1
9  01001      1       1


In [129]:
lm_df1['FIPS'] = lm_df1['FIPS_state'].astype(str).str.zfill(2) + lm_df1['FIPS_county'].astype(str).str.zfill(3)
print(lm_df1[['FIPS', 'FIPS_state', 'FIPS_county']].head(10))

    FIPS  FIPS_state  FIPS_county
0  01001           1            1
1  01003           1            3
2  01005           1            5
3  01007           1            7
4  01009           1            9
5  01011           1           11
6  01013           1           13
7  01015           1           15
8  01017           1           17
9  01019           1           19
